# Six-day NOAA-20 detections forecast for Spain

Run all cells in Colab. The default issue is **2026-11-15 00:00 UTC**, with daily scores for **15–20 November UTC**. The export uses the original 0.25° grid alignment and retains cells with positive area of overlap with the full Spain geometry stored in the map file. Coastal or border cells still represent full grid rectangles. The bundled ESP geometry comes from the same Kelvin367 `wwwroot/data/countries.json` used by the map. Set `COUNTRIES_JSON` to use an updated copy of that file.

The notebook loads the saved central Spain CatBoost run. Cells outside the original training grid are marked with `outside_training_grid: true`. Accuracy in those cells has not been evaluated. For 18–20 November, the second weather block reuses the trained Day 1–3 models and is also marked `experimental`.

- `forecast_grid(date, days=6)` returns dates, grid cells and probabilities indexed by **day, then cell**.
- `predict(date, latitude, longitude, days=6)` returns the daily series for a point inside the Spain boundary.
- `spain_forecast.json` uses the same schema as the previous grid export. With an input `spain.json`, the output preserves the weekly data and adds the daily forecast under `forecast`.

The target is the probability of a NOAA-20 FIRMS detection within the full 0.25° cell during the UTC day. Data sources and forecast coverage are recorded in the forecast metadata.


In [ ]:
# AI-assisted: generated with ChatGPT based on team's design.

In [ ]:
from collections import Counter
from functools import lru_cache
from hashlib import sha256
from importlib.util import find_spec
from math import ceil, floor
from pathlib import Path
from time import sleep
import json
import subprocess
import sys

from google.colab import drive, userdata
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/spain_firms_2012_2025")
ISSUE_DATE = "2026-11-15"  # Date of issue at 00:00 UTC.
REQUESTED_DAYS = 6
RUN_DIR = None            # None selects the latest complete NOAA-20 model export on Drive.
EE_PROJECT = ""           # Or use the EE_PROJECT Colab secret.
COUNTRIES_JSON = None     # Optional path to use a newer Kelvin367 countries.json.
HISTORY_JSON = None       # Or set Path(...) to the supplied spain.json to append the forecast.
OUTPUT_PATH = None        # Or set Path(...) to an explicit export path on Drive.


def install_if_missing(packages):
    missing = [package for module, package in packages.items() if find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


install_if_missing({"catboost": "catboost>=1.2.8,<2", "pyarrow": "pyarrow"})
install_if_missing({"ee": "earthengine-api", "xclim": "xclim==0.59.1", "shapely": "shapely>=2,<3"})

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display
from shapely.geometry import Point, box, shape
from shapely.prepared import prep

RUN_ROOT = DRIVE_ROOT / "model_runs" / "noaa20_environment_catboost_3day"
HORIZONS = (1, 2, 3)
MODEL_FILES = {day: f"catboost_day{day}.cbm" for day in HORIZONS}
manifest_name = "model_manifest.json"
if RUN_DIR is None:
    complete_runs = [p for p in RUN_ROOT.glob("*") if (p / manifest_name).is_file()
                     and all((p / name).is_file() for name in MODEL_FILES.values())]
    if not complete_runs:
        raise FileNotFoundError(f"No complete CatBoost export in {RUN_ROOT}. Run the training notebook's save cell or set RUN_DIR.")
    RUN_DIR = max(complete_runs)
RUN_DIR = Path(RUN_DIR)
manifest = json.loads((RUN_DIR / manifest_name).read_text(encoding="utf-8"))
if manifest["target"].get("platform") != "NOAA20" or manifest.get("firms_usage") != "labels_only":
    raise ValueError("Select the NOAA-20 model export with FIRMS used only for labels.")
if manifest["forecast_window_start_offsets_days"] != {str(day): day - 1 for day in HORIZONS}:
    raise ValueError("The model export uses unexpected forecast windows.")

FEATURES = {day: manifest["features_by_lead"][str(day)] for day in HORIZONS}
all_features = list(dict.fromkeys(name for day in HORIZONS for name in FEATURES[day]))
models = {day: CatBoostClassifier().load_model(str(RUN_DIR / name)) for day, name in MODEL_FILES.items()}
for day, model in models.items():
    if model.feature_names_ != FEATURES[day]:
        raise ValueError(f"Day {day} model features disagree with the manifest. Use files from one training run.")

TRAINING_BBOX = manifest["bbox_wsen"]
CELL_DEG = manifest["cell_degrees"]
CELL_KEYS = ["cell_y", "cell_x"]
KEYS = ["issue_utc", *CELL_KEYS]
COUNTRIES_REPO_PATH = 'app/src/FireMap.Web/wwwroot/data/countries.json'
RUN_AGE_HOURS = 6
GFS_DATASET = "NOAA/GFS0P25"
GFS_BASE = ["temp_max_24h", "rh_min_24h", "wind_max_24h", "precip_sum_24h"]
MAX_FORECAST_DAYS = 2 * max(HORIZONS)
GFS = {day: [f"gfs_day{day}_{name}" for name in GFS_BASE] for day in range(1, MAX_FORECAST_DAYS + 1)}
TERRAIN = ["elevation_mean_m", "slope_mean_deg"]
LAND = ["forest_frac", "shrubland_frac", "natural_grassland_frac", "agriculture_frac"]
SMAP = ["smap_surface_daily", "smap_surface_mean_7d"]
FEATURE_DIR = DRIVE_ROOT / "features"
OUTPUT_ROOT = DRIVE_ROOT / "forecast_runs" / "noaa20_environment_catboost_spain"
MISSING_PIXEL = -9999.0
def utc(values):
    return pd.to_datetime(values, utc=True, errors="raise").astype("datetime64[ns, UTC]")


def secret(name):
    try:
        return (userdata.get(name) or "").strip()
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return ""


def align_cells(frame):
    indexed = frame.set_index(CELL_KEYS)
    if not indexed.index.is_unique or not indexed.index.isin(CELL_INDEX).all():
        raise ValueError("Input contains duplicate or out-of-grid cell keys.")
    return indexed.reindex(CELL_INDEX)


def numeric(values):
    parsed = pd.to_numeric(values, errors="raise").astype(float)
    return parsed.where(np.isfinite(parsed) & parsed.ne(MISSING_PIXEL))


def issue_timestamp(date):
    value = pd.to_datetime(date, utc=True, errors="raise")
    if pd.isna(value) or value != value.normalize():
        raise ValueError("Pass a UTC calendar date such as 2026-11-15.")
    if value > pd.Timestamp.now(tz="UTC"):
        raise ValueError("The requested 00:00 UTC issue time is still in the future.")
    return value.as_unit("ns")


print(f"Model export: {RUN_DIR.name}")


## Earth Engine, Spain grid and GFS weather

The GFS cycle for each issue is the preceding day's **18:00 UTC** cycle. All six dates use weather from that same cycle. The experimental block feeds weather for Days 4–6 into the saved models' Day 1–3 input slots and recomputes their cumulative forecast rain for that block. SMAP, NDMI, FWI, historical weather, season and static inputs are anchored to the original issue. Trailing predictors use Day 1 forecasts from their original issue dates. For forecasting on a different date, the notebook assembles the corresponding inputs again.


In [ ]:
import ee

project = EE_PROJECT.strip() or secret("EE_PROJECT")
if not project:
    raise ValueError("Set EE_PROJECT above or add it to Colab Secrets.")
try:
    ee.Initialize(project=project)
except ee.EEException:
    ee.Authenticate(auth_mode="notebook", force=True)
    ee.Initialize(project=project)


def spain_grid(boundary):
    prepared = prep(boundary)
    west, south, east, north = boundary.bounds
    pairs = []
    for y in range(floor(south / CELL_DEG), ceil(north / CELL_DEG)):
        for x in range(floor(west / CELL_DEG), ceil(east / CELL_DEG)):
            cell = box(x * CELL_DEG, y * CELL_DEG, (x + 1) * CELL_DEG, (y + 1) * CELL_DEG)
            if prepared.intersects(cell) and boundary.intersection(cell).area > 0:
                pairs.append((y, x))
    if not pairs:
        raise ValueError("The Spain boundary did not intersect the forecast grid.")
    return boundary, pd.MultiIndex.from_tuples(pairs, names=CELL_KEYS)


# ESP feature copied from Kelvin367-main.zip. The original country file is unchanged.
BUNDLED_SPAIN_FEATURE = json.loads(r'''{
  "type": "Feature",
  "geometry": {
    "type": "MultiPolygon",
    "coordinates": [
      [
        [
          [
            -5.378,
            35.882
          ],
          [
            -5.341,
            35.847
          ],
          [
            -5.283,
            35.912
          ],
          [
            -5.399,
            35.925
          ],
          [
            -5.378,
            35.882
          ]
        ]
      ],
      [
        [
          [
            -2.922,
            35.288
          ],
          [
            -2.948,
            35.33
          ],
          [
            -2.964,
            35.286
          ],
          [
            -2.913,
            35.277
          ],
          [
            -2.922,
            35.288
          ]
        ]
      ],
      [
        [
          [
            -1.797,
            43.341
          ],
          [
            -1.794,
            43.386
          ],
          [
            -1.926,
            43.343
          ],
          [
            -1.919,
            43.321
          ],
          [
            -1.96,
            43.343
          ],
          [
            -2.128,
            43.287
          ],
          [
            -2.367,
            43.316
          ],
          [
            -2.661,
            43.412
          ],
          [
            -2.691,
            43.368
          ],
          [
            -2.715,
            43.425
          ],
          [
            -2.769,
            43.453
          ],
          [
            -2.947,
            43.431
          ],
          [
            -3.029,
            43.376
          ],
          [
            -3.008,
            43.321
          ],
          [
            -3.269,
            43.404
          ],
          [
            -3.461,
            43.412
          ],
          [
            -3.502,
            43.431
          ],
          [
            -3.432,
            43.467
          ],
          [
            -3.579,
            43.518
          ],
          [
            -3.756,
            43.472
          ],
          [
            -3.745,
            43.438
          ],
          [
            -3.811,
            43.425
          ],
          [
            -3.844,
            43.445
          ],
          [
            -3.769,
            43.479
          ],
          [
            -3.827,
            43.497
          ],
          [
            -3.94,
            43.477
          ],
          [
            -3.981,
            43.438
          ],
          [
            -4.482,
            43.384
          ],
          [
            -5.178,
            43.489
          ],
          [
            -5.3,
            43.538
          ],
          [
            -5.414,
            43.507
          ],
          [
            -5.394,
            43.555
          ],
          [
            -5.677,
            43.561
          ],
          [
            -5.852,
            43.663
          ],
          [
            -5.907,
            43.637
          ],
          [
            -5.907,
            43.588
          ],
          [
            -6.069,
            43.57
          ],
          [
            -6.223,
            43.596
          ],
          [
            -6.469,
            43.575
          ],
          [
            -6.463,
            43.555
          ],
          [
            -6.51,
            43.547
          ],
          [
            -6.649,
            43.579
          ],
          [
            -6.944,
            43.566
          ],
          [
            -7.001,
            43.557
          ],
          [
            -7.045,
            43.479
          ],
          [
            -7.044,
            43.559
          ],
          [
            -7.271,
            43.561
          ],
          [
            -7.257,
            43.582
          ],
          [
            -7.367,
            43.68
          ],
          [
            -7.541,
            43.732
          ],
          [
            -7.616,
            43.689
          ],
          [
            -7.641,
            43.753
          ],
          [
            -7.692,
            43.727
          ],
          [
            -7.686,
            43.793
          ],
          [
            -7.819,
            43.697
          ],
          [
            -7.868,
            43.697
          ],
          [
            -7.858,
            43.668
          ],
          [
            -7.909,
            43.67
          ],
          [
            -7.847,
            43.718
          ],
          [
            -7.909,
            43.766
          ],
          [
            -8.059,
            43.709
          ],
          [
            -8.052,
            43.643
          ],
          [
            -8.075,
            43.64
          ],
          [
            -8.087,
            43.656
          ],
          [
            -8.232,
            43.576
          ],
          [
            -8.313,
            43.567
          ],
          [
            -8.292,
            43.533
          ],
          [
            -8.333,
            43.472
          ],
          [
            -8.169,
            43.492
          ],
          [
            -8.305,
            43.451
          ],
          [
            -8.175,
            43.416
          ],
          [
            -8.21,
            43.391
          ],
          [
            -8.21,
            43.313
          ],
          [
            -8.267,
            43.386
          ],
          [
            -8.331,
            43.406
          ],
          [
            -8.354,
            43.349
          ],
          [
            -8.388,
            43.386
          ],
          [
            -8.558,
            43.313
          ],
          [
            -8.693,
            43.293
          ],
          [
            -8.83,
            43.341
          ],
          [
            -8.988,
            43.283
          ],
          [
            -8.933,
            43.235
          ],
          [
            -9.012,
            43.239
          ],
          [
            -9.052,
            43.196
          ],
          [
            -9.138,
            43.202
          ],
          [
            -9.218,
            43.155
          ],
          [
            -9.108,
            43.141
          ],
          [
            -9.218,
            43.114
          ],
          [
            -9.28,
            43.055
          ],
          [
            -9.251,
            43.017
          ],
          [
            -9.292,
            42.929
          ],
          [
            -9.273,
            42.893
          ],
          [
            -9.196,
            42.949
          ],
          [
            -9.135,
            42.918
          ],
          [
            -9.141,
            42.857
          ],
          [
            -9.1,
            42.833
          ],
          [
            -9.135,
            42.813
          ],
          [
            -9.1,
            42.758
          ],
          [
            -8.888,
            42.831
          ],
          [
            -8.898,
            42.792
          ],
          [
            -9.03,
            42.708
          ],
          [
            -9.071,
            42.596
          ],
          [
            -9.026,
            42.543
          ],
          [
            -8.903,
            42.625
          ],
          [
            -8.869,
            42.617
          ],
          [
            -8.84,
            42.682
          ],
          [
            -8.785,
            42.641
          ],
          [
            -8.731,
            42.688
          ],
          [
            -8.776,
            42.605
          ],
          [
            -8.823,
            42.57
          ],
          [
            -8.827,
            42.462
          ],
          [
            -8.881,
            42.497
          ],
          [
            -8.943,
            42.469
          ],
          [
            -8.844,
            42.404
          ],
          [
            -8.655,
            42.441
          ],
          [
            -8.772,
            42.343
          ],
          [
            -8.831,
            42.342
          ],
          [
            -8.827,
            42.297
          ],
          [
            -8.853,
            42.305
          ],
          [
            -8.868,
            42.256
          ],
          [
            -8.675,
            42.299
          ],
          [
            -8.623,
            42.347
          ],
          [
            -8.625,
            42.302
          ],
          [
            -8.847,
            42.161
          ],
          [
            -8.827,
            42.131
          ],
          [
            -8.895,
            42.119
          ],
          [
            -8.881,
            41.893
          ],
          [
            -8.671,
            42
          ],
          [
            -8.627,
            42.051
          ],
          [
            -8.222,
            42.154
          ],
          [
            -8.204,
            42.07
          ],
          [
            -8.138,
            42.073
          ],
          [
            -8.095,
            42.041
          ],
          [
            -8.23,
            41.905
          ],
          [
            -8.177,
            41.85
          ],
          [
            -8.18,
            41.811
          ],
          [
            -8.049,
            41.816
          ],
          [
            -7.906,
            41.914
          ],
          [
            -7.897,
            41.858
          ],
          [
            -7.722,
            41.899
          ],
          [
            -7.607,
            41.873
          ],
          [
            -7.606,
            41.826
          ],
          [
            -7.516,
            41.865
          ],
          [
            -7.469,
            41.853
          ],
          [
            -7.443,
            41.806
          ],
          [
            -7.219,
            41.879
          ],
          [
            -7.2,
            41.964
          ],
          [
            -7.145,
            41.987
          ],
          [
            -7.051,
            41.942
          ],
          [
            -6.992,
            41.966
          ],
          [
            -6.855,
            41.943
          ],
          [
            -6.825,
            41.984
          ],
          [
            -6.728,
            41.931
          ],
          [
            -6.609,
            41.962
          ],
          [
            -6.572,
            41.943
          ],
          [
            -6.576,
            41.883
          ],
          [
            -6.524,
            41.867
          ],
          [
            -6.577,
            41.74
          ],
          [
            -6.542,
            41.659
          ],
          [
            -6.366,
            41.664
          ],
          [
            -6.206,
            41.57
          ],
          [
            -6.321,
            41.411
          ],
          [
            -6.379,
            41.397
          ],
          [
            -6.496,
            41.295
          ],
          [
            -6.647,
            41.268
          ],
          [
            -6.804,
            41.064
          ],
          [
            -6.907,
            41.056
          ],
          [
            -6.942,
            41.016
          ],
          [
            -6.816,
            40.857
          ],
          [
            -6.839,
            40.758
          ],
          [
            -6.814,
            40.647
          ],
          [
            -6.85,
            40.582
          ],
          [
            -6.813,
            40.515
          ],
          [
            -6.857,
            40.442
          ],
          [
            -6.794,
            40.356
          ],
          [
            -6.878,
            40.265
          ],
          [
            -7.016,
            40.226
          ],
          [
            -7.043,
            40.181
          ],
          [
            -7.016,
            40.119
          ],
          [
            -6.956,
            40.103
          ],
          [
            -6.88,
            40.009
          ],
          [
            -6.928,
            39.862
          ],
          [
            -7,
            39.803
          ],
          [
            -7.021,
            39.694
          ],
          [
            -7.557,
            39.68
          ],
          [
            -7.516,
            39.594
          ],
          [
            -7.314,
            39.457
          ],
          [
            -7.327,
            39.341
          ],
          [
            -7.251,
            39.271
          ],
          [
            -7.258,
            39.211
          ],
          [
            -7.168,
            39.171
          ],
          [
            -7.157,
            39.105
          ],
          [
            -7.016,
            39.097
          ],
          [
            -6.973,
            39.014
          ],
          [
            -7.066,
            38.905
          ],
          [
            -7.056,
            38.855
          ],
          [
            -7.27,
            38.738
          ],
          [
            -7.293,
            38.656
          ],
          [
            -7.27,
            38.618
          ],
          [
            -7.359,
            38.446
          ],
          [
            -7.167,
            38.272
          ],
          [
            -7.104,
            38.174
          ],
          [
            -6.948,
            38.197
          ],
          [
            -7.024,
            38.023
          ],
          [
            -7.123,
            38.04
          ],
          [
            -7.16,
            37.997
          ],
          [
            -7.273,
            37.977
          ],
          [
            -7.307,
            37.851
          ],
          [
            -7.444,
            37.73
          ],
          [
            -7.472,
            37.637
          ],
          [
            -7.514,
            37.601
          ],
          [
            -7.529,
            37.557
          ],
          [
            -7.469,
            37.476
          ],
          [
            -7.414,
            37.193
          ],
          [
            -7.086,
            37.206
          ],
          [
            -7.127,
            37.215
          ],
          [
            -7.101,
            37.224
          ],
          [
            -6.928,
            37.172
          ],
          [
            -6.963,
            37.234
          ],
          [
            -6.853,
            37.295
          ],
          [
            -6.918,
            37.215
          ],
          [
            -6.893,
            37.167
          ],
          [
            -6.5,
            36.96
          ],
          [
            -6.388,
            36.808
          ],
          [
            -6.347,
            36.82
          ],
          [
            -6.345,
            36.901
          ],
          [
            -6.195,
            36.932
          ],
          [
            -6.223,
            36.902
          ],
          [
            -6.338,
            36.884
          ],
          [
            -6.346,
            36.795
          ],
          [
            -6.433,
            36.751
          ],
          [
            -6.395,
            36.637
          ],
          [
            -6.283,
            36.612
          ],
          [
            -6.235,
            36.576
          ],
          [
            -6.224,
            36.511
          ],
          [
            -6.179,
            36.514
          ],
          [
            -6.231,
            36.473
          ],
          [
            -6.31,
            36.534
          ],
          [
            -6.159,
            36.304
          ],
          [
            -6.105,
            36.281
          ],
          [
            -6.037,
            36.19
          ],
          [
            -5.917,
            36.184
          ],
          [
            -5.796,
            36.086
          ],
          [
            -5.689,
            36.067
          ],
          [
            -5.612,
            36.006
          ],
          [
            -5.439,
            36.063
          ],
          [
            -5.425,
            36.18
          ],
          [
            -5.339,
            36.141
          ],
          [
            -5.291,
            36.277
          ],
          [
            -5.187,
            36.411
          ],
          [
            -4.903,
            36.507
          ],
          [
            -4.659,
            36.508
          ],
          [
            -4.588,
            36.574
          ],
          [
            -4.525,
            36.581
          ],
          [
            -4.403,
            36.724
          ],
          [
            -4.225,
            36.712
          ],
          [
            -3.839,
            36.752
          ],
          [
            -3.731,
            36.724
          ],
          [
            -3.601,
            36.743
          ],
          [
            -3.415,
            36.696
          ],
          [
            -3.33,
            36.745
          ],
          [
            -2.933,
            36.752
          ],
          [
            -2.848,
            36.7
          ],
          [
            -2.718,
            36.682
          ],
          [
            -2.617,
            36.728
          ],
          [
            -2.561,
            36.819
          ],
          [
            -2.362,
            36.837
          ],
          [
            -2.204,
            36.737
          ],
          [
            -2.127,
            36.736
          ],
          [
            -1.989,
            36.9
          ],
          [
            -1.9,
            36.941
          ],
          [
            -1.906,
            36.988
          ],
          [
            -1.813,
            37.207
          ],
          [
            -1.671,
            37.363
          ],
          [
            -1.497,
            37.428
          ],
          [
            -1.471,
            37.482
          ],
          [
            -1.324,
            37.562
          ],
          [
            -1.243,
            37.579
          ],
          [
            -1.106,
            37.532
          ],
          [
            -1.12,
            37.559
          ],
          [
            -1.088,
            37.581
          ],
          [
            -0.925,
            37.556
          ],
          [
            -0.705,
            37.62
          ],
          [
            -0.726,
            37.652
          ],
          [
            -0.81,
            37.667
          ],
          [
            -0.856,
            37.742
          ],
          [
            -0.788,
            37.823
          ],
          [
            -0.761,
            37.796
          ],
          [
            -0.718,
            37.943
          ],
          [
            -0.655,
            37.989
          ],
          [
            -0.624,
            38.162
          ],
          [
            -0.504,
            38.207
          ],
          [
            -0.51,
            38.338
          ],
          [
            -0.414,
            38.365
          ],
          [
            -0.329,
            38.47
          ],
          [
            -0.075,
            38.54
          ],
          [
            -0.043,
            38.612
          ],
          [
            0.081,
            38.631
          ],
          [
            0.083,
            38.661
          ],
          [
            0.235,
            38.742
          ],
          [
            0.154,
            38.831
          ],
          [
            -0.01,
            38.87
          ],
          [
            -0.158,
            39.002
          ],
          [
            -0.227,
            39.112
          ],
          [
            -0.242,
            39.166
          ],
          [
            -0.215,
            39.182
          ],
          [
            -0.301,
            39.322
          ],
          [
            -0.323,
            39.516
          ],
          [
            -0.091,
            39.848
          ],
          [
            0.205,
            40.172
          ],
          [
            0.351,
            40.285
          ],
          [
            0.563,
            40.59
          ],
          [
            0.733,
            40.634
          ],
          [
            0.663,
            40.575
          ],
          [
            0.592,
            40.582
          ],
          [
            0.616,
            40.553
          ],
          [
            0.677,
            40.565
          ],
          [
            0.784,
            40.661
          ],
          [
            0.858,
            40.686
          ],
          [
            0.874,
            40.728
          ],
          [
            0.767,
            40.785
          ],
          [
            0.695,
            40.786
          ],
          [
            0.714,
            40.815
          ],
          [
            1.017,
            41.052
          ],
          [
            1.173,
            41.059
          ],
          [
            1.218,
            41.105
          ],
          [
            1.539,
            41.187
          ],
          [
            2.131,
            41.304
          ],
          [
            2.267,
            41.449
          ],
          [
            2.933,
            41.712
          ],
          [
            3.079,
            41.827
          ],
          [
            3.15,
            41.845
          ],
          [
            3.232,
            41.946
          ],
          [
            3.196,
            42.033
          ],
          [
            3.214,
            42.068
          ],
          [
            3.122,
            42.14
          ],
          [
            3.119,
            42.218
          ],
          [
            3.17,
            42.26
          ],
          [
            3.263,
            42.24
          ],
          [
            3.318,
            42.323
          ],
          [
            3.187,
            42.35
          ],
          [
            3.181,
            42.431
          ],
          [
            2.934,
            42.47
          ],
          [
            2.69,
            42.406
          ],
          [
            2.64,
            42.373
          ],
          [
            2.662,
            42.339
          ],
          [
            2.548,
            42.352
          ],
          [
            2.515,
            42.326
          ],
          [
            2.277,
            42.429
          ],
          [
            2.137,
            42.416
          ],
          [
            2.009,
            42.347
          ],
          [
            1.964,
            42.367
          ],
          [
            1.927,
            42.437
          ],
          [
            1.707,
            42.503
          ],
          [
            1.517,
            42.43
          ],
          [
            1.448,
            42.435
          ],
          [
            1.408,
            42.487
          ],
          [
            1.45,
            42.504
          ],
          [
            1.406,
            42.529
          ],
          [
            1.429,
            42.595
          ],
          [
            1.343,
            42.709
          ],
          [
            1.151,
            42.707
          ],
          [
            1.089,
            42.771
          ],
          [
            0.921,
            42.784
          ],
          [
            0.813,
            42.832
          ],
          [
            0.656,
            42.838
          ],
          [
            0.633,
            42.745
          ],
          [
            0.657,
            42.688
          ],
          [
            0.409,
            42.682
          ],
          [
            0.354,
            42.717
          ],
          [
            0.275,
            42.669
          ],
          [
            0.169,
            42.726
          ],
          [
            -0.039,
            42.685
          ],
          [
            -0.119,
            42.717
          ],
          [
            -0.161,
            42.795
          ],
          [
            -0.235,
            42.786
          ],
          [
            -0.323,
            42.843
          ],
          [
            -0.424,
            42.791
          ],
          [
            -0.534,
            42.808
          ],
          [
            -0.569,
            42.773
          ],
          [
            -0.648,
            42.85
          ],
          [
            -0.74,
            42.885
          ],
          [
            -0.76,
            42.947
          ],
          [
            -0.943,
            42.948
          ],
          [
            -1.15,
            43.006
          ],
          [
            -1.294,
            43.055
          ],
          [
            -1.315,
            43.083
          ],
          [
            -1.286,
            43.109
          ],
          [
            -1.348,
            43.093
          ],
          [
            -1.366,
            43.033
          ],
          [
            -1.457,
            43.045
          ],
          [
            -1.488,
            43.082
          ],
          [
            -1.426,
            43.135
          ],
          [
            -1.404,
            43.243
          ],
          [
            -1.559,
            43.284
          ],
          [
            -1.622,
            43.247
          ],
          [
            -1.655,
            43.298
          ],
          [
            -1.747,
            43.291
          ],
          [
            -1.797,
            43.341
          ]
        ]
      ],
      [
        [
          [
            1.948,
            42.451
          ],
          [
            1.999,
            42.444
          ],
          [
            1.957,
            42.482
          ],
          [
            1.948,
            42.451
          ]
        ]
      ],
      [
        [
          [
            -17.879,
            27.815
          ],
          [
            -17.934,
            27.852
          ],
          [
            -18.057,
            27.756
          ],
          [
            -18.167,
            27.758
          ],
          [
            -18.16,
            27.722
          ],
          [
            -17.979,
            27.643
          ],
          [
            -17.879,
            27.815
          ]
        ]
      ],
      [
        [
          [
            -15.364,
            28.01
          ],
          [
            -15.392,
            28.03
          ],
          [
            -15.409,
            28.18
          ],
          [
            -15.459,
            28.134
          ],
          [
            -15.706,
            28.167
          ],
          [
            -15.708,
            28.082
          ],
          [
            -15.816,
            27.996
          ],
          [
            -15.821,
            27.922
          ],
          [
            -15.74,
            27.804
          ],
          [
            -15.578,
            27.734
          ],
          [
            -15.378,
            27.852
          ],
          [
            -15.364,
            28.01
          ]
        ]
      ],
      [
        [
          [
            -17.213,
            28.205
          ],
          [
            -17.31,
            28.207
          ],
          [
            -17.343,
            28.098
          ],
          [
            -17.258,
            28.025
          ],
          [
            -17.185,
            28.032
          ],
          [
            -17.098,
            28.101
          ],
          [
            -17.116,
            28.146
          ],
          [
            -17.213,
            28.205
          ]
        ]
      ],
      [
        [
          [
            -16.546,
            28.024
          ],
          [
            -16.428,
            28.15
          ],
          [
            -16.356,
            28.381
          ],
          [
            -16.118,
            28.572
          ],
          [
            -16.316,
            28.572
          ],
          [
            -16.507,
            28.424
          ],
          [
            -16.77,
            28.366
          ],
          [
            -16.837,
            28.391
          ],
          [
            -16.916,
            28.355
          ],
          [
            -16.693,
            28
          ],
          [
            -16.546,
            28.024
          ]
        ]
      ],
      [
        [
          [
            -13.865,
            28.744
          ],
          [
            -13.973,
            28.736
          ],
          [
            -14.224,
            28.219
          ],
          [
            -14.341,
            28.135
          ],
          [
            -14.497,
            28.103
          ],
          [
            -14.507,
            28.071
          ],
          [
            -14.343,
            28.051
          ],
          [
            -14.192,
            28.171
          ],
          [
            -13.934,
            28.244
          ],
          [
            -13.843,
            28.407
          ],
          [
            -13.824,
            28.682
          ],
          [
            -13.865,
            28.744
          ]
        ]
      ],
      [
        [
          [
            -17.721,
            28.731
          ],
          [
            -17.756,
            28.819
          ],
          [
            -17.911,
            28.856
          ],
          [
            -17.997,
            28.78
          ],
          [
            -17.837,
            28.455
          ],
          [
            -17.76,
            28.577
          ],
          [
            -17.765,
            28.678
          ],
          [
            -17.721,
            28.731
          ]
        ]
      ],
      [
        [
          [
            -13.418,
            29.209
          ],
          [
            -13.483,
            29.24
          ],
          [
            -13.537,
            29.138
          ],
          [
            -13.637,
            29.127
          ],
          [
            -13.798,
            29.048
          ],
          [
            -13.867,
            28.865
          ],
          [
            -13.775,
            28.84
          ],
          [
            -13.709,
            28.915
          ],
          [
            -13.482,
            28.999
          ],
          [
            -13.459,
            29.147
          ],
          [
            -13.418,
            29.209
          ]
        ]
      ],
      [
        [
          [
            -13.492,
            29.243
          ],
          [
            -13.508,
            29.289
          ],
          [
            -13.541,
            29.229
          ],
          [
            -13.492,
            29.243
          ]
        ]
      ],
      [
        [
          [
            -0.74,
            37.702
          ],
          [
            -0.755,
            37.774
          ],
          [
            -0.747,
            37.684
          ],
          [
            -0.726,
            37.665
          ],
          [
            -0.74,
            37.702
          ]
        ]
      ],
      [
        [
          [
            1.566,
            38.691
          ],
          [
            1.526,
            38.678
          ],
          [
            1.431,
            38.755
          ],
          [
            1.383,
            38.712
          ],
          [
            1.383,
            38.642
          ],
          [
            1.458,
            38.681
          ],
          [
            1.581,
            38.653
          ],
          [
            1.566,
            38.691
          ]
        ]
      ],
      [
        [
          [
            1.623,
            39.029
          ],
          [
            1.606,
            39.091
          ],
          [
            1.523,
            39.119
          ],
          [
            1.357,
            39.078
          ],
          [
            1.293,
            39.033
          ],
          [
            1.301,
            38.968
          ],
          [
            1.218,
            38.954
          ],
          [
            1.217,
            38.885
          ],
          [
            1.403,
            38.832
          ],
          [
            1.415,
            38.891
          ],
          [
            1.623,
            39.029
          ]
        ]
      ],
      [
        [
          [
            3.474,
            39.75
          ],
          [
            3.355,
            39.784
          ],
          [
            3.224,
            39.75
          ],
          [
            3.136,
            39.789
          ],
          [
            3.126,
            39.832
          ],
          [
            3.207,
            39.893
          ],
          [
            3.095,
            39.881
          ],
          [
            3.097,
            39.914
          ],
          [
            3.214,
            39.955
          ],
          [
            3.179,
            39.969
          ],
          [
            2.813,
            39.873
          ],
          [
            2.358,
            39.612
          ],
          [
            2.351,
            39.576
          ],
          [
            2.392,
            39.524
          ],
          [
            2.454,
            39.537
          ],
          [
            2.493,
            39.47
          ],
          [
            2.569,
            39.532
          ],
          [
            2.682,
            39.559
          ],
          [
            2.744,
            39.523
          ],
          [
            2.737,
            39.462
          ],
          [
            2.787,
            39.376
          ],
          [
            2.959,
            39.363
          ],
          [
            3.061,
            39.269
          ],
          [
            3.253,
            39.386
          ],
          [
            3.323,
            39.524
          ],
          [
            3.454,
            39.66
          ],
          [
            3.474,
            39.75
          ]
        ]
      ],
      [
        [
          [
            4.337,
            39.867
          ],
          [
            4.171,
            40.065
          ],
          [
            4.142,
            40.025
          ],
          [
            4.105,
            40.079
          ],
          [
            3.822,
            40.048
          ],
          [
            3.803,
            40.018
          ],
          [
            3.851,
            39.984
          ],
          [
            3.831,
            39.936
          ],
          [
            3.981,
            39.936
          ],
          [
            4.248,
            39.816
          ],
          [
            4.3,
            39.822
          ],
          [
            4.303,
            39.888
          ],
          [
            4.337,
            39.867
          ]
        ]
      ],
      [
        [
          [
            2.962,
            39.116
          ],
          [
            2.981,
            39.186
          ],
          [
            2.914,
            39.152
          ],
          [
            2.962,
            39.116
          ]
        ]
      ],
      [
        [
          [
            -4.299,
            35.173
          ],
          [
            -4.302,
            35.173
          ],
          [
            -4.297,
            35.172
          ],
          [
            -4.299,
            35.173
          ]
        ]
      ],
      [
        [
          [
            -3.9,
            35.218
          ],
          [
            -3.901,
            35.217
          ],
          [
            -3.9,
            35.217
          ],
          [
            -3.9,
            35.218
          ]
        ]
      ],
      [
        [
          [
            -2.429,
            35.186
          ],
          [
            -2.431,
            35.179
          ],
          [
            -2.427,
            35.182
          ],
          [
            -2.429,
            35.186
          ]
        ]
      ],
      [
        [
          [
            -5.416,
            35.914
          ],
          [
            -5.421,
            35.914
          ],
          [
            -5.418,
            35.911
          ],
          [
            -5.416,
            35.914
          ]
        ]
      ]
    ]
  },
  "properties": {
    "iso3": "ESP",
    "name": "Spain"
  }
}''')
BUNDLED_COUNTRIES_SHA256 = "ba3650402bd2e4108fcd38b6f0d71c881723ff2cc5fdc6372046cd6781107a90"


def load_spain_boundary(path):
    if path is None:
        return BUNDLED_SPAIN_FEATURE["geometry"], BUNDLED_COUNTRIES_SHA256
    raw = Path(path).read_bytes()
    countries = json.loads(raw)
    matching = [
        feature for feature in countries["features"]
        if feature.get("properties", {}).get("iso3") == "ESP"
    ]
    if len(matching) != 1:
        raise ValueError("Expected exactly one country feature with iso3=ESP in COUNTRIES_JSON.")
    return matching[0]["geometry"], sha256(raw).hexdigest()


spain_geometry, countries_sha256 = load_spain_boundary(COUNTRIES_JSON)
SPAIN, CELL_INDEX = spain_grid(shape(spain_geometry))
grid_cells = CELL_INDEX.to_frame(index=False)
ys = np.arange(grid_cells.cell_y.min(), grid_cells.cell_y.max() + 1)
xs = np.arange(grid_cells.cell_x.min(), grid_cells.cell_x.max() + 1)
WEST, SOUTH, EAST, NORTH = [float(value * CELL_DEG) for value in (xs[0], ys[0], xs[-1] + 1, ys[-1] + 1)]
train_west, train_south, train_east, train_north = TRAINING_BBOX
inside_training = (
    ((grid_cells.cell_x + 0.5) * CELL_DEG).between(train_west, train_east, inclusive="left")
    & ((grid_cells.cell_y + 0.5) * CELL_DEG).between(train_south, train_north, inclusive="left")
)
outside_training = ~inside_training.to_numpy()
GRID_ID = sha256(grid_cells.to_numpy(dtype="<i4").tobytes()).hexdigest()[:12]
transform = [CELL_DEG, 0, WEST, 0, -CELL_DEG, NORTH]
pixel_grid = {
    "dimensions": {"width": len(xs), "height": len(ys)},
    "affineTransform": dict(zip(
        ["scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY"], transform,
    )),
    "crsCode": "EPSG:4326",
}
pixel_rows = ys[-1] - grid_cells.cell_y.to_numpy()
pixel_cols = grid_cells.cell_x.to_numpy() - xs[0]
print(f"Spain grid: {len(CELL_INDEX):,} cells, {int(outside_training.sum()):,} outside the training grid.")


In [ ]:
GFS_STATE_BANDS = [
    "temperature_2m_above_ground", "relative_humidity_2m_above_ground",
    "u_component_of_wind_10m_above_ground", "v_component_of_wind_10m_above_ground",
]
GFS_RAIN_BAND = "total_precipitation_surface"


def weather_leads(day):
    first = RUN_AGE_HOURS + 24 * (day - 1)
    return tuple(range(first, first + 24, 3)), tuple(range(first + 6, first + 25, 6))


def gfs_day_image(archive, issue, day):
    run = issue - pd.Timedelta(hours=RUN_AGE_HOURS)
    cycle = archive.filter(ee.Filter.eq("creation_time", run.value // 10**6))
    state_leads, rain_leads = weather_leads(day)

    def step(lead, bands, names):
        valid = run + pd.Timedelta(hours=lead)
        matching = cycle.filter(ee.Filter.eq("forecast_hours", lead)).filter(
            ee.Filter.eq("forecast_time", valid.value // 10**6)
        )
        missing = ee.Image.constant([0] * len(names)).rename(names).selfMask().toFloat()
        image = ee.Image(matching.first()).select(bands, names).toFloat().resample("bilinear")
        image = image.reproject(crs="EPSG:4326", crsTransform=transform)
        return ee.Image(ee.Algorithms.If(matching.size().eq(1), image, missing))

    def state(lead):
        inputs = step(lead, GFS_STATE_BANDS, ["temperature", "humidity", "u", "v"])
        wind = inputs.select("u").hypot(inputs.select("v")).rename("wind")
        return inputs.select(["temperature", "humidity"]).addBands(wind)

    def reduce_complete(collection, field, reducer, count):
        band = collection.select(field)
        return getattr(band, reducer)().updateMask(band.count().eq(count))

    states = ee.ImageCollection.fromImages([state(lead) for lead in state_leads])
    rain = ee.ImageCollection.fromImages([step(lead, [GFS_RAIN_BAND], ["rain"]) for lead in rain_leads])
    images = [
        reduce_complete(states, "temperature", "max", len(state_leads)),
        reduce_complete(states, "humidity", "min", len(state_leads)),
        reduce_complete(states, "wind", "max", len(state_leads)),
        reduce_complete(rain, "rain", "sum", len(rain_leads)),
    ]
    return ee.Image.cat(*images).rename(GFS[day]).toFloat().unmask(MISSING_PIXEL)



def compute_pixels(image, label):
    request = {"expression": image, "fileFormat": "NUMPY_NDARRAY", "grid": pixel_grid}
    for attempt in range(3):
        try:
            pixels = ee.data.computePixels(request)
            break
        except ee.EEException as error:
            retryable = any(token in str(error).lower() for token in (
                "429", "500", "502", "503", "504", "deadline", "temporarily", "timed out",
            ))
            if attempt == 2 or not retryable:
                raise
            sleep(2 ** (attempt + 1))
    if pixels.shape != (len(ys), len(xs)):
        raise ValueError(f"Unexpected {label} grid shape: {pixels.shape}.")
    return pixels


def fetch_weather(days, forecast_days):
    archive = ee.ImageCollection(GFS_DATASET)
    issue = days[-1]
    run = issue - pd.Timedelta(hours=RUN_AGE_HOURS)
    cycle = archive.filter(ee.Filter.eq("creation_time", run.value // 10**6))
    leads = Counter(int(lead) for lead in cycle.aggregate_array("forecast_hours").getInfo())
    if not leads:
        raise RuntimeError(f"GFS cycle {run:%Y-%m-%d %H:%M} UTC is unavailable in Earth Engine.")
    for day in forecast_days:
        missing = sorted(lead for lead in set().union(*weather_leads(day)) if leads[lead] != 1)
        if missing:
            print(f"Day {day} missing or duplicate GFS forecast hours: {missing}")

    tables = []
    for start in range(0, len(days), 4):
        block = days[start:start + 4]
        images, names = [], {}
        for date in block:
            leads_for_date = forecast_days if date == issue else (1,)
            names[date] = [name for day in leads_for_date for name in GFS[day]]
            images.extend(
                gfs_day_image(archive, date, day).rename([f"d{date:%Y%m%d}_{name}" for name in GFS[day]])
                for day in leads_for_date
            )
        pixels = compute_pixels(ee.Image.cat(*images), "GFS")
        for date in block:
            table = grid_cells.assign(issue_utc=date)
            for name in names[date]:
                values = numeric(pd.Series(pixels[f"d{date:%Y%m%d}_{name}"][pixel_rows, pixel_cols]))
                table[name] = values.where(values.ge(-273.15)) if "temp_max" in name else values.clip(lower=0)
            tables.append(table)
    return pd.concat(tables, ignore_index=True)


## Terrain, land cover and moisture history

SMAP keeps the original eight-day offset from issue date and a complete seven-day trailing mean. Terra NDMI uses the original composite QA, minimum valid coverage and release buffer. Observation dates are retained for the models that were trained with observation-age features.


Terrain and land cover reuse the model input tables where available. The notebook downloads NASADEM and CORINE features for the remaining cells using the training resolutions and land-cover groups.


In [ ]:
def cell_regions(ee, cells):
    return ee.FeatureCollection([
        ee.Feature(
            ee.Geometry.Rectangle(
                [x * CELL_DEG, y * CELL_DEG, (x + 1) * CELL_DEG, (y + 1) * CELL_DEG],
                proj="EPSG:4326",
                geodesic=False,
            ),
            {"cell_y": int(y), "cell_x": int(x)},
        )
        for y, x in cells[CELL_KEYS].itertuples(index=False, name=None)
    ])


def fetch_smap_block(ee, smap, projection, regions, cells, days):
    # Keep the band present on archive days with no usable frames.
    masked = ee.Image.constant(0).rename("sm_surface").toFloat().selfMask()
    fallback = ee.ImageCollection.fromImages([masked])

    images = []
    band_dates = {}
    for day in days:
        start = day.strftime("%Y-%m-%d")
        end = (day + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        day_id = day.strftime("d%Y%m%d")
        frames = smap.filterDate(start, end).merge(fallback)
        daily = frames.mean().updateMask(frames.count().eq(8))
        images.append(daily.toFloat().set("system:index", day_id))
        band_dates[f"{day_id}_sm_surface"] = day

    stats = ee.ImageCollection.fromImages(images).toBands().reduceRegions(
        collection=regions,
        reducer=ee.Reducer.mean(),
        crs=projection["crs"],
        crsTransform=projection["transform"],
        tileScale=4,
    )
    wide = ee.data.computeFeatures({"expression": stats, "fileFormat": "PANDAS_DATAFRAME"})
    expected_cells = pd.MultiIndex.from_frame(cells)
    returned_cells = pd.MultiIndex.from_frame(wide[CELL_KEYS])
    if not returned_cells.is_unique or not expected_cells.sort_values().equals(returned_cells.sort_values()):
        raise ValueError("SMAP grid keys differ from the forecast grid.")

    daily = wide.reindex(columns=[*CELL_KEYS, *band_dates]).melt(
        id_vars=CELL_KEYS, var_name="band", value_name="sm_surface",
    )
    daily[SMAP_SOURCE_DAY] = daily.pop("band").map(band_dates)
    daily["sm_surface"] = pd.to_numeric(daily["sm_surface"], errors="raise")
    return daily


def ndmi_image(image):
    state = image.select("StateQA")
    quality = image.select("QA")
    nir = image.select("sur_refl_b02")
    swir = image.select("sur_refl_b06")

    cloud_and_snow_flags = (1 << 2) | (3 << 8) | (1 << 10) | (1 << 12) | (1 << 13) | (1 << 15)
    band_quality_flags = (15 << 6) | (15 << 22)
    valid = (
        state.bitwiseAnd(3).eq(0)
        .And(state.rightShift(3).bitwiseAnd(7).eq(1))
        .And(state.bitwiseAnd(cloud_and_snow_flags).eq(0))
        .And(quality.bitwiseAnd(3).lt(2))
        .And(quality.bitwiseAnd(band_quality_flags).eq(0))
        .And(nir.gte(0)).And(nir.lte(16000))
        .And(swir.gte(0)).And(swir.lte(16000))
        .And(nir.add(swir).gt(0))
    )
    ndmi = (
        image.normalizedDifference(["sur_refl_b02", "sur_refl_b06"])
        .updateMask(valid)
        .rename("ndmi_mean")
    )
    coverage = ndmi.mask().unmask(0, sameFootprint=False).rename(NDMI_COVERAGE)
    return ndmi.addBands(coverage).toFloat()


def fetch_ndmi_block(ee, modis, projection, regions, cells, block):
    images = []
    bands_by_day = {}
    # Name the output bands directly, independently of MODIS asset IDs.
    for day in block:
        start = day.strftime("%Y-%m-%d")
        stop = (day + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        image = ee.Image(modis.filterDate(start, stop).first())
        prefix = day.strftime("d%Y%m%d_")
        names = [prefix + "ndmi_mean", prefix + NDMI_COVERAGE]
        bands_by_day[day] = names
        images.append(ndmi_image(image).rename(names))

    stats = ee.Image.cat(*images).reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(),
        crs=projection["crs"], crsTransform=projection["transform"],
        tileScale=4,
    )
    wide = ee.data.computeFeatures({"expression": stats, "fileFormat": "PANDAS_DATAFRAME"})
    expected = pd.MultiIndex.from_frame(cells).sort_values()
    received = pd.MultiIndex.from_frame(wide[CELL_KEYS]).sort_values()
    if not received.equals(expected):
        raise ValueError("Earth Engine returned different NDMI cell keys.")

    parts = []
    for day, (mean_column, coverage_column) in bands_by_day.items():
        if coverage_column not in wide.columns:
            raise ValueError(
                f"Earth Engine did not return {coverage_column}. "
                f"Returned columns: {wide.columns.tolist()}"
            )
        coverage = pd.to_numeric(wide[coverage_column])
        mean = pd.to_numeric(wide.get(mean_column, pd.Series(float("nan"), index=wide.index)))
        if not coverage.between(0, 1).all():
            raise ValueError(f"Invalid NDMI coverage in {coverage_column}.")
        if (coverage.gt(0) & mean.isna()).any():
            raise ValueError(f"NDMI coverage is positive without a value for {mean_column}.")
        parts.append(wide[CELL_KEYS].assign(**{
            NDMI_SOURCE_DATE: day,
            "ndmi_mean": mean,
            NDMI_COVERAGE: coverage,
        }))
    return pd.concat(parts, ignore_index=True)


In [ ]:
SMAP_SOURCE_DAY = "smap_observation_day_utc"
NDMI_SOURCE_DATE = "ndmi_composite_start_utc"
NDMI_COVERAGE = "ndmi_valid_fraction"



COVER_GROUPS = {
    "forest_frac": [311, 312, 313],
    "shrubland_frac": [322, 323, 324],
    "natural_grassland_frac": [321],
    "agriculture_frac": [211, 212, 213, 221, 222, 223, 231, 241, 242, 243, 244],
}


def align_saved_static(frame):
    return align_cells(frame.loc[pd.MultiIndex.from_frame(frame[CELL_KEYS]).isin(CELL_INDEX)])


def fill_static(frame, image, label, *, scale, crs=None):
    needed = frame.index[frame.isna().all(axis=1)]
    if needed.empty:
        return frame
    cache = OUTPUT_ROOT / "cache" / f"static_{label}_{GRID_ID}.parquet"
    if cache.is_file():
        existing = align_cells(pd.read_parquet(cache))
    else:
        existing = pd.DataFrame(np.nan, index=CELL_INDEX, columns=frame.columns)
    needed = needed.difference(existing.index[existing.notna().any(axis=1)])
    tables = [existing.dropna(how="all")]
    for i in range(0, len(needed), 160):
        block = needed[i:i + 160].to_frame(index=False)
        print(f"{label}: cells {i + 1}–{i + len(block)} of {len(needed)}", flush=True)
        options = {"crs": crs} if crs is not None else {}
        reduced = image.reduceRegions(
            collection=cell_regions(ee, block), reducer=ee.Reducer.mean(), scale=scale,
            tileScale=4, **options,
        )
        result = ee.data.computeFeatures({"expression": reduced, "fileFormat": "PANDAS_DATAFRAME"})
        table = align_cells(result.reindex(columns=[*CELL_KEYS, *frame.columns]))
        tables.append(table.dropna(how="all"))

    available = pd.concat(tables).reindex(CELL_INDEX)
    if len(tables) > 1 and available.notna().any(axis=None):
        cache.parent.mkdir(parents=True, exist_ok=True)
        available.dropna(how="all").reset_index().to_parquet(cache, index=False)
    missing = frame.isna().all(axis=1)
    frame.loc[missing] = available.loc[missing]
    return frame


def load_static(columns, snapshot_file, fallback=True):
    path = FEATURE_DIR / snapshot_file
    required = [*CELL_KEYS, *columns]
    if path.is_file():
        return align_saved_static(pd.read_parquet(path, columns=required))
    combined = FEATURE_DIR / "environmental_features.parquet"
    if fallback and combined.is_file():
        from pyarrow.parquet import read_schema

        if set(required).issubset(read_schema(combined).names):
            saved = pd.read_parquet(combined, columns=required).groupby(CELL_KEYS, as_index=False)
            if saved.nunique()[columns].gt(1).any(axis=None):
                raise ValueError(f"Use the static input {snapshot_file}; its values vary in the combined table.")
            return align_saved_static(saved.first())
    return pd.DataFrame(np.nan, columns=columns, index=CELL_INDEX)


def static_features(issue):
    terrain = load_static(TERRAIN, "elevation_by_cell.parquet").apply(numeric)
    dem = ee.Image("NASA/NASADEM_HGT/001").select("elevation")
    terrain_image = dem.addBands(ee.Terrain.slope(dem)).rename(TERRAIN)
    terrain = fill_static(terrain, terrain_image, "nasadem", scale=90).apply(numeric)
    terrain["slope_mean_deg"] = terrain["slope_mean_deg"].where(terrain["slope_mean_deg"].between(0, 90))

    land = pd.DataFrame(np.nan, columns=LAND, index=CELL_INDEX)
    for vintage, release in sorted(manifest["corine_release_dates_used"].items(), reverse=True):
        if issue >= pd.Timestamp(release, tz="UTC"):
            land = load_static(LAND, f"corine_{vintage}_by_cell.parquet", fallback=vintage == "2018")
            clc = ee.Image(f"COPERNICUS/CORINE/V20/100m/{vintage}").select("landcover")
            cover_bands = ee.Image.cat(*[
                clc.remap(classes, [1] * len(classes), defaultValue=0).rename(name)
                for name, classes in COVER_GROUPS.items()
            ])
            land = fill_static(land.apply(numeric), cover_bands, f"corine_{vintage}", scale=100, crs="EPSG:3035")
            break
    land = land.apply(numeric).where(lambda values: values.ge(0) & values.le(1))
    return pd.concat([terrain, land], axis=1)


def history_index(days):
    return pd.MultiIndex.from_arrays([
        days.repeat(len(CELL_INDEX)),
        np.tile(grid_cells.cell_y.to_numpy(), len(days)),
        np.tile(grid_cells.cell_x.to_numpy(), len(days)),
    ], names=KEYS)


def fetch_smap_history(days, regions):
    obs_days = pd.date_range(days[0] - pd.Timedelta(days=14), days[-1] - pd.Timedelta(days=8), freq="D")
    smap = (ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").select("sm_surface")
            .filterDate(obs_days[0].isoformat(), (obs_days[-1] + pd.Timedelta(days=1)).isoformat()))
    dense = pd.DataFrame(np.nan, index=obs_days, columns=CELL_INDEX)
    if smap.size().getInfo():
        projection = ee.Image(smap.first()).projection().getInfo()
        daily = fetch_smap_block(ee, smap, projection, regions, grid_cells, obs_days)
        daily[SMAP_SOURCE_DAY] = utc(daily[SMAP_SOURCE_DAY])
        dense = daily.pivot(index=SMAP_SOURCE_DAY, columns=CELL_KEYS, values="sm_surface").reindex(
            index=obs_days, columns=CELL_INDEX,
        )
        dense = dense.where(dense.ge(0) & dense.le(1))
    source_days = days - pd.Timedelta(days=8)
    index = history_index(days)
    return pd.DataFrame({
        SMAP[0]: dense.loc[source_days].to_numpy().ravel(),
        SMAP[1]: dense.rolling(7).mean().loc[source_days].to_numpy().ravel(),
        SMAP_SOURCE_DAY: source_days.repeat(len(CELL_INDEX)),
    }, index=index)


def fetch_ndmi_history(days):
    delay = pd.Timedelta(days=11)
    fallback = pd.Timedelta(days=24)
    latest, earliest = days[-1] - delay, days[0] - delay - fallback
    modis = (ee.ImageCollection("MODIS/061/MOD09A1")
             .filterDate(earliest.isoformat(), (latest + pd.Timedelta(days=1)).isoformat())
             .select(["sur_refl_b02", "sur_refl_b06", "StateQA", "QA"]))
    dates = pd.DatetimeIndex(pd.to_datetime(
        modis.aggregate_array("system:time_start").getInfo(), unit="ms", utc=True,
    )).as_unit("ns").sort_values()
    idx = history_index(days)
    if dates.empty:
        return pd.DataFrame({"ndmi_mean": np.nan, NDMI_SOURCE_DATE: pd.NaT}, index=idx)
    if not dates.is_unique or not dates.equals(dates.normalize()):
        raise ValueError("Expected unique MODIS composite dates at midnight.")
    projection = ee.Image(modis.first()).select("sur_refl_b02").projection().getInfo()
    tables = []
    for start in range(0, len(grid_cells), 192):
        cells_block = grid_cells.iloc[start:start + 192]
        regions_block = cell_regions(ee, cells_block)
        print(f"NDMI: cells {start + 1}–{start + len(cells_block)} of {len(grid_cells)}", flush=True)
        tables.append(fetch_ndmi_block(ee, modis, projection, regions_block, cells_block, dates))
    table = pd.concat(tables, ignore_index=True)
    table[CELL_KEYS] = table[CELL_KEYS].astype(grid_cells.dtypes)
    table[NDMI_SOURCE_DATE] = utc(table[NDMI_SOURCE_DATE])
    table["available_utc"] = table[NDMI_SOURCE_DATE] + delay
    usable = table.loc[table[NDMI_COVERAGE].between(0.20, 1) & table["ndmi_mean"].between(-1, 1)]
    matched = pd.merge_asof(
        idx.to_frame(index=False).sort_values("issue_utc"),
        usable.sort_values("available_utc"),
        left_on="issue_utc", right_on="available_utc", by=CELL_KEYS,
        direction="backward", tolerance=fallback,
    )
    return matched.set_index(KEYS)[["ndmi_mean", NDMI_SOURCE_DATE]].reindex(idx)


## Fire Weather Index

The live calculation uses the same GFS noon-weather recipe and `xclim` equations as training. It starts 120 days before the earliest FWI needed for the issue, so its moisture state can differ slightly from the continuously initialized training archive. Complete weather blocks are cached on Drive for reuse.


In [ ]:
def clean_gfs_values(frame, names):
    temperature = names[0]
    for name in names:
        values = pd.to_numeric(frame[name], errors="raise")
        invalid = np.isinf(values)
        if name == temperature:
            invalid |= values.lt(-273.15)
            negative = pd.Series(False, index=values.index)
        else:
            negative = values.lt(0) & ~invalid

        flagged = invalid | negative
        if flagged.any():
            dates = frame.loc[flagged, "issue_utc"]
            original = values[flagged]
            print(
                f"GFS quality {name} {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}: "
                f"{int(negative.sum())} clipped to zero, {int(invalid.sum())} set to NaN; "
                f"original values {original.min():.6g} to {original.max():.6g}.",
                flush=True,
            )
        frame[name] = values.mask(invalid).mask(negative, 0)
    return frame


def fwi_day_image(ee, archive, issue):
    run_18 = issue - pd.Timedelta(hours=RUN_AGE_HOURS)
    noon = issue + pd.Timedelta(hours=12)

    def select_at(valid, latest_run, source_bands, names):
        candidates = []
        for backup_hours in (24, 6, 0):
            run = latest_run - pd.Timedelta(hours=backup_hours)
            lead = int((valid - run) / pd.Timedelta(hours=1))
            matching = (archive.filter(ee.Filter.eq("creation_time", run.value // 10**6))
                        .filter(ee.Filter.eq("forecast_time", valid.value // 10**6))
                        .filter(ee.Filter.eq("forecast_hours", lead)))
            empty = ee.Image.constant([0] * len(names)).rename(names).selfMask().toFloat()
            forecast = (ee.Image(matching.first()).select(source_bands, names).toFloat()
                        .resample("bilinear").reproject(crs="EPSG:4326", crsTransform=transform))
            forecast = forecast.updateMask(forecast.mask().reduce(ee.Reducer.min()))
            candidates.append(ee.Image(ee.Algorithms.If(matching.size().eq(1), forecast, empty)))
        return ee.ImageCollection.fromImages(candidates).mosaic()

    state = select_at(
        noon, run_18, [*FWI_GFS_BANDS, FWI_GFS_RAIN], ["temperature", "humidity", "u", "v", "rain"],
    )
    wind = state.select("u").hypot(state.select("v")).multiply(3.6)
    rain_bands = [FWI_GFS_RAIN]
    rain_intervals = [
        select_at(run_18, run_18 - pd.Timedelta(hours=6), rain_bands, ["rain"]),
        *[select_at(run_18 + pd.Timedelta(hours=lead), run_18, rain_bands, ["rain"])
          for lead in (6, 12)],
        state.select("rain"),
    ]
    # Adding the bands directly keeps the daily total masked if any interval is missing.
    rain = rain_intervals[0].add(rain_intervals[1]).add(rain_intervals[2]).add(rain_intervals[3])
    bands = state.select(["temperature", "humidity"]).addBands([wind, rain])
    return bands.toFloat().unmask(MISSING_PIXEL).rename([
        f"d{issue:%Y%m%d}_{name}" for name in FWI_WEATHER
    ])


def fetch_fwi_weather(ee, block):
    archive = ee.ImageCollection(GFS_DATASET).filter(ee.Filter.inList(
        "forecast_hours", list(range(6, 43, 6)),
    ))
    stack = ee.Image.cat(*[fwi_day_image(ee, archive, day) for day in block])
    pixels = compute_pixels(stack, "FWI")
    frames = []
    for day in block:
        frame = grid_cells.assign(issue_utc=day)
        for name in FWI_WEATHER:
            values = pixels[f"d{day:%Y%m%d}_{name}"][pixel_rows, pixel_cols].astype(np.float32)
            frame[name] = np.where(values == MISSING_PIXEL, np.nan, values)
        frames.append(frame[[*KEYS, *FWI_WEATHER]])
    return pd.concat(frames, ignore_index=True)


def calculate_fwi(weather, days):
    import xarray as xr
    from xclim.indices.fire import cffwis_indices

    weather = weather.assign(issue_utc=pd.to_datetime(weather["issue_utc"], utc=True))
    weather = clean_gfs_values(weather, FWI_WEATHER).set_index(KEYS, verify_integrity=True)
    grid = history_index(days)
    if len(grid.difference(weather.index)) or len(weather.index.difference(grid)):
        raise ValueError("FWI weather does not cover the expected dates and cells.")

    values = weather.reindex(grid)[FWI_WEATHER].to_numpy(dtype=float, copy=True).reshape(len(days), -1, 4)
    # The FWI equations include fractional powers of (100 - RH).
    values[:, :, 1] = np.minimum(values[:, :, 1], 100)
    valid = np.isfinite(values).all(axis=2)
    fwi = np.full(valid.shape, np.nan, dtype=np.float32)
    latitudes = (grid_cells.cell_y.to_numpy() + 0.5) * CELL_DEG
    dates = days.tz_localize(None)

    # Group cells with the same missing days so each run can use contiguous valid weather.
    _, groups = np.unique(valid.T, axis=0, return_inverse=True)
    for group in np.unique(groups):
        cells = np.flatnonzero(groups == group)
        usable_days = np.flatnonzero(valid[:, cells[0]])
        segments = np.split(usable_days, np.flatnonzero(np.diff(usable_days) > 1) + 1)
        previous_codes = {}
        lat = xr.DataArray(latitudes[cells], dims="cell", attrs={"units": "degrees_north"})
        for segment in segments:
            if not len(segment):
                continue
            data = {
                arg: xr.DataArray(values[segment[:, None], cells, idx], dims=("time", "cell"),
                                  coords={"time": dates[segment]}, attrs={"units": unit})
                for idx, (arg, unit) in enumerate((
                    ("tas", "degC"), ("hurs", "%"), ("sfcWind", "km/h"), ("pr", "mm/day"),
                ))
            }
            indices = cffwis_indices(**data, lat=lat, season_method=None, **previous_codes)
            fwi[np.ix_(segment, cells)] = indices.FWI.transpose("time", "cell").values
            # Missing days retain the last modeled moisture state and have no FWI value.
            previous_codes = {name.lower() + "0": getattr(indices, name).isel(time=-1, drop=True).copy()
                              for name in ("FFMC", "DMC", "DC")}

    return pd.DataFrame({"fwi": fwi.ravel()}, index=grid).reset_index()

FWI_GFS_BANDS = GFS_STATE_BANDS
FWI_GFS_RAIN = GFS_RAIN_BAND
FWI_WEATHER = ["fwi_temp_c", "fwi_rh_pct", "fwi_wind_kmh", "fwi_rain_mm"]


def fetch_fwi_history(history_days):
    need = history_days[-8:]
    days = pd.date_range(need[0] - pd.Timedelta(days=120), need[-1], freq="D")
    signature = sha256(repr((GFS_DATASET, GRID_ID, transform, "noon_utc_v1")).encode()).hexdigest()[:12]
    cache_dir = OUTPUT_ROOT / "cache" / f"fwi_{signature}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    parts = []
    # Fixed calendar blocks let forecasts issued on different days reuse the cache.
    cache_anchor = pd.Timestamp("2015-07-02", tz="UTC")
    block_start = days[0] - pd.Timedelta(days=(days[0] - cache_anchor).days % 14)
    for start in pd.date_range(block_start, days[-1], freq="14D"):
        end = min(start + pd.Timedelta(days=13), days[-1])
        block = pd.date_range(start, end, freq="D")
        path = cache_dir / f"{start:%Y%m%d}_{end:%Y%m%d}.parquet"
        if path.is_file():
            part = pd.read_parquet(path)
        else:
            print(f"GFS for FWI · {start:%Y-%m-%d} to {end:%Y-%m-%d}", flush=True)
            part = fetch_fwi_weather(ee, block)
            if part[FWI_WEATHER].notna().all(axis=None):
                part.to_parquet(path, index=False)
        parts.append(part)
    weather = pd.concat(parts, ignore_index=True)
    scores = calculate_fwi(weather.loc[utc(weather["issue_utc"]).isin(days)].copy(), days)
    return scores.set_index(KEYS)[["fwi"]].reindex(history_index(history_days))


## Assemble the saved model's feature columns

This cell applies the training notebook's trailing weather windows, prior FWI mean, changes in SMAP and NDMI, observation ages, and cumulative forecast rain. It checks the feature names and order against the saved model manifest.


In [ ]:
def build_temporal_features(panel, panel_index):
    # Each matrix row is a UTC issue day. Each column is one fixed grid cell.
    if not pd.MultiIndex.from_frame(panel[KEYS]).equals(panel_index):
        raise ValueError("Temporal features require the full, ordered daily grid.")

    n_cells = len(CELL_INDEX)
    values = {}

    def matrix(column):
        return pd.DataFrame(panel[column].to_numpy(dtype=np.float64).reshape(-1, n_cells))

    def add(column, daily_grid):
        values[column] = np.asarray(daily_grid, dtype=np.float32).ravel()

    # Historical weather summaries use Day 1 forecasts from past issues.
    # Shift first so today's target-day weather is outside these windows.
    for column, extreme in zip(GFS[1][:3], ("max", "min", "max")):
        previous = matrix(column).shift(1)
        for window, min_days in ((3, 2), (7, 5)):
            history = previous.rolling(window, min_periods=min_days)
            add(f"{column}_mean_prior_{window}d", history.mean())
            add(f"{column}_{extreme}_prior_{window}d", getattr(history, extreme)())

    previous_rain = matrix(GFS[1][-1]).shift(1)
    for window in (3, 7, 14):
        add(f"gfs_precip_sum_prior_{window}d", previous_rain.rolling(window).sum())

    previous_fwi = matrix("fwi").shift(1)
    add("fwi_mean_prior_7d", previous_fwi.rolling(7, min_periods=5).mean())

    for column, lag in (("smap_surface_daily", 7), ("ndmi_mean", 8)):
        series = matrix(column)
        add(f"{column}_change_{lag}d", series - series.shift(lag))


    for column, date_column in ((SMAP[0], SMAP_SOURCE_DAY), ("ndmi_mean", NDMI_SOURCE_DATE)):
        age = (panel["issue_utc"] - utc(panel[date_column])).dt.days
        add(f"{column}_age_days", age.where(panel[column].notna() & age.ge(0)))

    for lead in HORIZONS[1:]:
        rain = [GFS[day][-1] for day in range(1, lead + 1)]
        add(f"gfs_rain_through_day{lead}", panel[rain].sum(axis=1, min_count=lead))
    return pd.DataFrame(values, index=panel.index, copy=False)


def build_inputs(issue, include_experimental):
    days = pd.date_range(issue - pd.Timedelta(days=14), issue, freq="D")
    idx = history_index(days)
    warnings = []
    forecast_days = tuple(GFS) if include_experimental else HORIZONS
    weather = fetch_weather(days, forecast_days).set_index(KEYS, verify_integrity=True).reindex(idx)
    static = static_features(issue).reindex(idx.droplevel("issue_utc")).set_axis(idx)
    regions = cell_regions(ee, grid_cells)
    tables = [weather, static]
    for label, load, fallback in (
        ("SMAP", lambda: fetch_smap_history(days, regions), {SMAP[0]: np.nan, SMAP[1]: np.nan, SMAP_SOURCE_DAY: pd.NaT}),
        ("Terra NDMI", lambda: fetch_ndmi_history(days), {"ndmi_mean": np.nan, NDMI_SOURCE_DATE: pd.NaT}),
        ("FWI", lambda: fetch_fwi_history(days), {"fwi": np.nan}),
    ):
        try:
            tables.append(load())
        except ee.EEException as error:
            warnings.append(f"{label} unavailable from Earth Engine.")
            print(f"{label}: {error}")
            tables.append(pd.DataFrame(fallback, index=idx))

    panel = pd.concat(tables, axis=1).reset_index()
    panel["lat_center"] = ((panel["cell_y"] + 0.5) * CELL_DEG).astype(np.float32)
    panel["lon_center"] = ((panel["cell_x"] + 0.5) * CELL_DEG).astype(np.float32)
    phase = 2 * np.pi * (panel["issue_utc"].dt.dayofyear - 1) / 365.25
    panel["season_sin"] = np.sin(phase).astype(np.float32)
    panel["season_cos"] = np.cos(phase).astype(np.float32)
    panel = pd.concat([panel, build_temporal_features(panel, idx)], axis=1)
    missing = sorted(set(all_features).difference(panel.columns))
    if missing:
        raise ValueError(f"Inference is missing training columns: {missing}")
    rows = panel.loc[panel["issue_utc"].eq(issue)].set_index(CELL_KEYS, verify_integrity=True)
    inputs = rows[all_features].apply(numeric).astype(np.float32).reindex(CELL_INDEX)
    extended = None
    if include_experimental:
        extended = inputs.copy()
        offset = max(HORIZONS)
        for horizon in HORIZONS:
            extended[GFS[horizon]] = rows[GFS[horizon + offset]].apply(numeric).astype(np.float32).set_axis(
                GFS[horizon], axis=1,
            )
            if horizon > 1:
                rain = [GFS[earlier][-1] for earlier in range(1, horizon + 1)]
                extended[f"gfs_rain_through_day{horizon}"] = extended[rain].sum(axis=1, min_count=horizon)
    return inputs, extended, warnings


## Date-and-location interface and JSON export

The output's `days[i]` corresponds to `series.noaa20_detection_probability[i]`. For the grid output, each row of that matrix follows the order of `cells`. The point output has one probability per day. All dates use UTC. Probabilities are 0–1.

The arrays `quality.method_by_day` and `quality.model_horizon_by_day` follow the same date order. `experimental` identifies scores extrapolated beyond the saved models' trained horizons.

Each cell has an `outside_training_grid` flag. This spatial flag applies to all six dates. The temporal `experimental` flag applies to Days 4–6 throughout the grid.


In [ ]:
SERIES = "noaa20_detection_probability"
CELL_META = [
    {
        "cell_y": int(y), "cell_x": int(x),
        "outside_training_grid": bool(outside),
        "lat_center": float((y + 0.5) * CELL_DEG),
        "lon_center": float((x + 0.5) * CELL_DEG),
        "bbox_wsen": [float(x * CELL_DEG), float(y * CELL_DEG),
                      float((x + 1) * CELL_DEG), float((y + 1) * CELL_DEG)],
    }
    for (y, x), outside in zip(CELL_INDEX, outside_training)
]


def score_grid(inputs):
    scores = {}
    for lead in HORIZONS:
        weather_present = inputs[GFS[lead]].notna().any(axis=1)
        probability = pd.Series(np.nan, index=CELL_INDEX, dtype=float)
        if weather_present.any():
            probability.loc[weather_present] = models[lead].predict_proba(
                inputs.loc[weather_present, FEATURES[lead]]
            )[:, 1]
        scores[lead] = [float(value) if pd.notna(value) else None for value in probability]
    return scores


@lru_cache(maxsize=2)
def score_issue(issue, include_experimental):
    inputs, extended, warnings = build_inputs(issue, include_experimental)
    scores = score_grid(inputs)
    if extended is not None:
        scores.update({lead + max(HORIZONS): row for lead, row in score_grid(extended).items()})
    empty = [name for name in all_features if inputs[name].isna().all()]
    if empty:
        warnings.append("Some model features are missing across the entire grid. See missing_model_features.")
    return scores, empty, warnings


def forecast_grid(date, days=6):
    if isinstance(days, bool) or not isinstance(days, (int, np.integer)) or days < 1:
        raise ValueError("days must be a positive integer.")
    days = int(days)
    issue = issue_timestamp(date)
    native_days = max(HORIZONS)
    scores, empty, warnings = score_issue(issue, include_experimental=days > native_days)
    leads = range(1, min(days, MAX_FORECAST_DAYS) + 1)
    dates = [(issue + pd.Timedelta(days=lead - 1)).strftime("%Y-%m-%d") for lead in leads]
    matrix = [scores[lead].copy() for lead in leads]
    scored = [sum(value is not None for value in row) for row in matrix]
    return {
        "id": "spain",
        "name": "Spain",
        "kind": "daily_grid_forecast",
        "geometry_note": "Full 0.25 degree cells intersecting Spain as defined by the Kelvin367 map boundary.",
        "bbox_wsen": [WEST, SOUTH, EAST, NORTH],
        "cell_degrees": CELL_DEG,
        "cells": CELL_META,
        "days": dates,
        "series": {SERIES: matrix},
        "quality": {
            "unit": "probability (0–1)",
            "target": "At least one NOAA-20 FIRMS detection in the cell during the UTC day",
            "model_run": RUN_DIR.name,
            "training_grid_bbox_wsen": TRAINING_BBOX,
            "cells_outside_training_grid": int(outside_training.sum()),
            "boundary_source": {
                "path": COUNTRIES_REPO_PATH,
                "sha256": countries_sha256,
                "provider": "Kelvin367 map geometry (Natural Earth 1:10m, Ukraine point of view)",
                "selection": "All polygons in the ESP feature, as supplied",
            },
            "scored_cells_by_day": scored,
            "method_by_day": ["trained_horizon" if lead <= native_days else "experimental" for lead in leads],
            "model_horizon_by_day": [(lead - 1) % native_days + 1 for lead in leads],
            "missing_model_features": empty.copy(),
        },
        "data_policy": {
            "issue_utc": issue.isoformat(),
            "gfs_run_utc": (issue - pd.Timedelta(hours=RUN_AGE_HOURS)).isoformat(),
            "requested_days": days,
            "trained_max_days": native_days,
            "max_days_with_experimental": MAX_FORECAST_DAYS,
        },
        "warnings": [
            *warnings,
            "Scores outside the central Spain training grid have not been validated geographically.",
            *(["Days 4–6 reuse the saved models with later weather from the original GFS cycle. "
               "Their accuracy has not been validated at these horizons."] if days > native_days else []),
            *([f"This export supports at most {MAX_FORECAST_DAYS} days per issue."]
              if days > MAX_FORECAST_DAYS else []),
            *(["A null score means no GFS fields for that cell and horizon were available."]
              if any(count < len(CELL_INDEX) for count in scored) else []),
        ],
    }


def predict(date, latitude, longitude, days=6):
    latitude, longitude = float(latitude), float(longitude)
    if not (SOUTH <= latitude < NORTH and WEST <= longitude < EAST):
        raise ValueError(f"Location outside model grid: longitude [{WEST}, {EAST}), latitude [{SOUTH}, {NORTH}).")
    if not SPAIN.covers(Point(longitude, latitude)):
        raise ValueError("Coordinates must be inside the Spain boundary.")
    y, x = floor(latitude / CELL_DEG), floor(longitude / CELL_DEG)
    index = CELL_INDEX.get_indexer([(y, x)])[0]
    if index < 0:
        raise ValueError("The coordinates are on the edge of the retained Spain grid.")
    grid = forecast_grid(date, days)
    probabilities = [row[index] for row in grid["series"][SERIES]]
    quality = {**grid["quality"], "available_by_day": [value is not None for value in probabilities]}
    del quality["scored_cells_by_day"]
    return {
        "id": grid["id"],
        "name": grid["name"],
        "kind": "daily_point_forecast",
        "geometry_note": "Forecast for the grid cell containing the requested coordinates.",
        "location": {"latitude": latitude, "longitude": longitude, **CELL_META[index]},
        "days": grid["days"],
        "series": {SERIES: probabilities},
        "quality": quality,
        "data_policy": grid["data_policy"],
        "warnings": grid["warnings"],
    }


def save_forecast(forecast, history_path=None, output_path=None):
    if history_path is None:
        candidates = (root / name for root in (DRIVE_ROOT, DRIVE_ROOT.parent, Path("/content"))
                      for name in ("spain.json", "spain(1).json"))
        history_path = next((path for path in candidates if path.is_file()), None)
    if output_path is None:
        issue_date = forecast["data_policy"]["issue_utc"][:10]
        output_path = OUTPUT_ROOT / issue_date / "spain_forecast.json"
    output_path = Path(output_path)
    payload = forecast
    if history_path is not None:
        history_path = Path(history_path)
        if output_path.resolve() == history_path.resolve():
            raise ValueError("Choose a separate output file to preserve the original spain.json.")
        history = json.loads(history_path.read_text(encoding="utf-8"))
        if history.get("id") != "spain" or "weeks" not in history:
            raise ValueError("HISTORY_JSON must be the weekly Spain file.")
        payload = {**history, "forecast": forecast}
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    return output_path


In [ ]:
forecast = forecast_grid(ISSUE_DATE, days=REQUESTED_DAYS)
output = save_forecast(forecast, HISTORY_JSON, OUTPUT_PATH)
print(f"Saved {output}")
display(pd.DataFrame({
    "date_utc": forecast["days"],
    "method": forecast["quality"]["method_by_day"],
    "scored_cells": forecast["quality"]["scored_cells_by_day"],
    "total_cells": len(forecast["cells"]),
}))

# Point example:
# predict("2026-11-15", latitude=40.4168, longitude=-3.7038, days=6)
